In [1]:
import os
import json
import glob
from PIL import Image
from tqdm.notebook import tqdm
import re

In [2]:
SOURCE_IMAGES_DIR = "data/dataset"
ANNOTATIONS_DIR = "data/annotations"

OUTPUT_DATASET_DIR = "disease_dataset"

MODEL_FILTER = "gemini"

print("Настройки заданы.")

Настройки заданы.


In [3]:
def sanitize_filename(name: str) -> str:
    name = re.sub(r'\(.*?\)', '', name)
    name = name.strip().replace(' ', '_').replace('-', '_')
    name = re.sub(r'[^\w\d_]', '', name)
    return name

In [4]:
test_name = "рак (бактериальный)"
print(f"'{test_name}' -> '{sanitize_filename(test_name)}'")

'рак (бактериальный)' -> 'рак'


In [5]:
json_files = glob.glob(os.path.join(ANNOTATIONS_DIR, "*.json"))

if not json_files:
    print(f"В папке '{ANNOTATIONS_DIR}' не найдено JSON-файлов.")
else:
    print(f"Найдено {len(json_files)} файлов с аннотациями. Начинаем обработку...")

    if not os.path.exists(OUTPUT_DATASET_DIR):
        os.makedirs(OUTPUT_DATASET_DIR)
        print(f"Создана папка '{OUTPUT_DATASET_DIR}'")
        
    processed_count = 0
    
    for json_path in tqdm(json_files, desc="Обработка файлов"):
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            if data.get("model_name") != MODEL_FILTER:
                continue

            image_filename = data.get("img_path")
            if not image_filename:
                tqdm.write(f"ПРЕДУПРЕЖДЕНИЕ: В {os.path.basename(json_path)} отсутствует ключ 'img_path'. Пропускаем.")
                continue
                
            source_image_path = os.path.join(SOURCE_IMAGES_DIR, image_filename)
            if not os.path.exists(source_image_path):
                tqdm.write(f"ПРЕДУПРЕЖДЕНИЕ: Не найдено исходное изображение '{source_image_path}'. Пропускаем.")
                continue

            image = Image.open(source_image_path).convert("RGB")

            for i, ann in enumerate(data.get("annotations", [])):
                bbox = ann.get("bbox")
                diseases = ann.get("diseases")

                if not bbox or not diseases:
                    continue

                top_species_info = sorted(diseases, key=lambda x: x['probability'], reverse=True)[0]
                species_name = top_species_info.get("disease")
                
                if not species_name:
                    continue

                species_folder_name = sanitize_filename(species_name)
                
                class_dir = os.path.join(OUTPUT_DATASET_DIR, species_folder_name)
                if not os.path.exists(class_dir):
                    os.makedirs(class_dir)

                try:
                    crop_box = tuple(map(int, bbox))
                    cropped_image = image.crop(crop_box)
                    
                    if cropped_image.width == 0 or cropped_image.height == 0:
                        tqdm.write(f"ПРЕДУПРЕЖДЕНИЕ: Пустой bbox {crop_box} в файле {os.path.basename(json_path)}. Пропускаем.")
                        continue
                    
                    base_filename = os.path.splitext(image_filename)[0]
                    output_filename = f"{base_filename}_{i}.jpg"
                    output_path = os.path.join(class_dir, output_filename)

                    cropped_image.save(output_path, "JPEG", quality=95)
                    processed_count += 1
                    
                except Exception as crop_error:
                    tqdm.write(f"ОШИБКА при вырезке/сохранении для {os.path.basename(json_path)}: {crop_error}")

        except Exception as e:
            tqdm.write(f"ОШИБКА при чтении файла {os.path.basename(json_path)}: {e}")

    print(f"\nОбработка завершена. Всего сохранено {processed_count} изображений.")

Найдено 361 файлов с аннотациями. Начинаем обработку...
Создана папка 'disease_dataset'


Обработка файлов:   0%|          | 0/361 [00:00<?, ?it/s]


Обработка завершена. Всего сохранено 430 изображений.


In [6]:
OUTPUT_DATASET_DIR = "species_dataset"

In [8]:
json_files = glob.glob(os.path.join(ANNOTATIONS_DIR, "*.json"))

if not json_files:
    print(f"В папке '{ANNOTATIONS_DIR}' не найдено JSON-файлов.")
else:
    print(f"Найдено {len(json_files)} файлов с аннотациями. Начинаем обработку...")

    if not os.path.exists(OUTPUT_DATASET_DIR):
        os.makedirs(OUTPUT_DATASET_DIR)
        print(f"Создана папка '{OUTPUT_DATASET_DIR}'")
        
    processed_count = 0
    
    for json_path in tqdm(json_files, desc="Обработка файлов"):
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            if data.get("model_name") != MODEL_FILTER:
                continue

            image_filename = data.get("img_path")
            if not image_filename:
                tqdm.write(f"ПРЕДУПРЕЖДЕНИЕ: В {os.path.basename(json_path)} отсутствует ключ 'img_path'. Пропускаем.")
                continue
                
            source_image_path = os.path.join(SOURCE_IMAGES_DIR, image_filename)
            if not os.path.exists(source_image_path):
                tqdm.write(f"ПРЕДУПРЕЖДЕНИЕ: Не найдено исходное изображение '{source_image_path}'. Пропускаем.")
                continue

            image = Image.open(source_image_path).convert("RGB")

            for i, ann in enumerate(data.get("annotations", [])):
                bbox = ann.get("bbox")
                species_probabilities = ann.get("species_probabilities")

                if not bbox or not species_probabilities:
                    continue

                top_species_info = sorted(species_probabilities, key=lambda x: x['probability'], reverse=True)[0]
                species_name = top_species_info.get("species")
                
                if not species_name:
                    continue

                species_folder_name = sanitize_filename(species_name)
                
                class_dir = os.path.join(OUTPUT_DATASET_DIR, species_folder_name)
                if not os.path.exists(class_dir):
                    os.makedirs(class_dir)

                try:
                    crop_box = tuple(map(int, bbox))
                    cropped_image = image.crop(crop_box)
                    
                    if cropped_image.width == 0 or cropped_image.height == 0:
                        tqdm.write(f"ПРЕДУПРЕЖДЕНИЕ: Пустой bbox {crop_box} в файле {os.path.basename(json_path)}. Пропускаем.")
                        continue
                    
                    base_filename = os.path.splitext(image_filename)[0]
                    output_filename = f"{base_filename}_{i}.jpg"
                    output_path = os.path.join(class_dir, output_filename)

                    cropped_image.save(output_path, "JPEG", quality=95)
                    processed_count += 1
                    
                except Exception as crop_error:
                    tqdm.write(f"ОШИБКА при вырезке/сохранении для {os.path.basename(json_path)}: {crop_error}")

        except Exception as e:
            tqdm.write(f"ОШИБКА при чтении файла {os.path.basename(json_path)}: {e}")

    print(f"\nОбработка завершена. Всего сохранено {processed_count} изображений.")

Найдено 361 файлов с аннотациями. Начинаем обработку...


Обработка файлов:   0%|          | 0/361 [00:00<?, ?it/s]


Обработка завершена. Всего сохранено 430 изображений.


In [9]:
gemini_data_1 = "output_flash.json"
gemini_data_2 = "output.json"

with open(gemini_data_1, 'r', encoding='utf-8') as f:
    data_1 = json.load(f)

with open(gemini_data_2, 'r', encoding='utf-8') as f:
    data_2 = json.load(f)

In [14]:
data_1.extend(data_2)

In [16]:
file_names = [item['filename'] for item in data_1]

In [18]:
len(file_names)

564

In [74]:
import os
from collections import defaultdict
from tqdm import tqdm

SEARCH_DIR = "diseases/diseases_dataset"

def find_duplicate_filenames(search_dir: str):
    """
    Ищет файлы с одинаковыми именами в разных поддиректориях.
    """
    if not os.path.isdir(search_dir):
        print(f"ОШИБКА: Директория '{search_dir}' не найдена.")
        return

    # Используем defaultdict для удобного сбора путей
    # Ключ: имя файла (например, "001.jpg")
    # Значение: список путей, где этот файл встречается
    filenames_map = defaultdict(list)
    
    print(f"Сканирование директории '{search_dir}'...")

    # Рекурсивно проходим по всем файлам
    # tqdm поможет видеть прогресс, если файлов очень много
    file_iterator = list(os.walk(search_dir))
    for root, _, files in tqdm(file_iterator, desc="Анализ папок"):
        for filename in files:
            # Добавляем полный путь к файлу в список для этого имени
            filenames_map[filename].append(root)
            
    print("Сканирование завершено. Поиск дубликатов...")
    
    # --- Поиск и вывод дубликатов ---
    duplicates_found = False
    for filename, paths in filenames_map.items():
        # Если в списке больше одного пути, значит, имя файла - дубликат
        if len(paths) > 1:
            if not duplicates_found:
                print("\n--- Найдены файлы с совпадающими именами: ---")
                duplicates_found = True
            
            print(f"\nФайл: '{filename}' найден в {len(paths)} местах:")
            # Выводим пути, где был найден файл
            for path in paths:
                print(f"  - {path}")

    if not duplicates_found:
        print("\nПроверка завершена. Файлов с совпадающими именами не найдено.")

find_duplicate_filenames(SEARCH_DIR)

Сканирование директории 'diseases/diseases_dataset'...


Анализ папок: 100%|██████████| 19/19 [00:00<00:00, 18987.80it/s]

Сканирование завершено. Поиск дубликатов...

Проверка завершена. Файлов с совпадающими именами не найдено.
